In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                           f1_score, roc_auc_score, confusion_matrix, 
                           classification_report)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks # type: ignore
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Load preprocessed data
print("Loading preprocessed data...")
X_train = np.load('../data/X_train.npy')
X_test = np.load('../data/X_test.npy')
y_train = np.load('../data/y_train.npy')
y_test = np.load('../data/y_test.npy')

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# ============================================
# INTENTIONALLY CREATE AN OVERFITTING MODEL
# ============================================

def create_overfit_nn(input_shape):
    """Create a deliberately overcomplex neural network that will overfit"""
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        # Too many layers and units
        layers.Dense(256, activation='relu', kernel_regularizer=keras.regularizers.l1_l2(l1=0.0001, l2=0.0001)),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        
        layers.Dense(32, activation='relu'),
        layers.BatchNormalization(),
        
        layers.Dense(16, activation='relu'),
        
        # Output layer
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    
    return model

# Create and train the overfitting model
print("\n" + "="*60)
print("TRAINING DELIBERATELY OVERCOMPLEX NEURAL NETWORK")
print("="*60)

input_shape = (X_train.shape[1],)
overfit_model = create_overfit_nn(input_shape)

overfit_model.summary()

# Train without early stopping and with many epochs
print("\nTraining model for 100 epochs (intentionally too many)...")
history = overfit_model.fit(
    X_train, y_train,
    validation_split=0.1,  # Small validation split
    epochs=100,
    batch_size=32,
    verbose=1
)

# ============================================
# EVALUATE THE OVERFIT MODEL
# ============================================

print("\n" + "="*60)
print("EVALUATING OVERFIT MODEL")
print("="*60)

# Training predictions
y_train_pred_proba = overfit_model.predict(X_train)
y_train_pred = (y_train_pred_proba > 0.5).astype(int)

# Test predictions
y_test_pred_proba = overfit_model.predict(X_test)
y_test_pred = (y_test_pred_proba > 0.5).astype(int)

# Calculate metrics
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

train_auc = roc_auc_score(y_train, y_train_pred_proba)
test_auc = roc_auc_score(y_test, y_test_pred_proba)

print(f"\nTraining Accuracy:  {train_accuracy:.4f} ({train_accuracy*100:.1f}%)")
print(f"Test Accuracy:      {test_accuracy:.4f} ({test_accuracy*100:.1f}%)")
print(f"Accuracy Gap:       {train_accuracy - test_accuracy:.4f}")

print(f"\nTraining AUC:       {train_auc:.4f}")
print(f"Test AUC:           {test_auc:.4f}")
print(f"AUC Gap:            {train_auc - test_auc:.4f}")

# Detailed classification report
print("\n" + "-"*40)
print("CLASSIFICATION REPORT (TEST SET)")
print("-"*40)
print(classification_report(y_test, y_test_pred, target_names=['Low Risk', 'High Risk']))

# ============================================
# VISUALIZE THE OVERFITTING
# ============================================

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 1. Training vs Validation Accuracy
axes[0, 0].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[0, 0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0, 0].set_title('Model Accuracy: Clear Overfitting', fontweight='bold', color='red')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Training vs Validation Loss
axes[0, 1].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0, 1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0, 1].set_title('Model Loss: Diverging Patterns', fontweight='bold', color='red')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Accuracy Comparison Bar Chart
models_comparison = ['Training', 'Test']
accuracy_values = [train_accuracy, test_accuracy]

bars = axes[0, 2].bar(models_comparison, accuracy_values, 
                      color=['#2ecc71', '#e74c3c'], alpha=0.7)
axes[0, 2].set_title('Accuracy: Train vs Test', fontweight='bold')
axes[0, 2].set_ylabel('Accuracy')
axes[0, 2].set_ylim(0, 1.0)

# Add value labels on bars
for bar, value in zip(bars, accuracy_values):
    height = bar.get_height()
    axes[0, 2].text(bar.get_x() + bar.get_width()/2., height + 0.02,
                   f'{value:.3f}', ha='center', va='bottom', fontweight='bold')

# 4. Confusion Matrix (Test Set)
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', 
            xticklabels=['Pred Low', 'Pred High'],
            yticklabels=['Actual Low', 'Actual High'],
            ax=axes[1, 0])
axes[1, 0].set_title('Confusion Matrix (Test Set)', fontweight='bold')

# 5. ROC Curve
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y_test, y_test_pred_proba)
axes[1, 1].plot(fpr, tpr, color='darkorange', lw=2, 
                label=f'ROC curve (AUC = {test_auc:.3f})')
axes[1, 1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1, 1].set_xlim([0.0, 1.0])
axes[1, 1].set_ylim([0.0, 1.05])
axes[1, 1].set_xlabel('False Positive Rate')
axes[1, 1].set_ylabel('True Positive Rate')
axes[1, 1].set_title('ROC Curve', fontweight='bold')
axes[1, 1].legend(loc="lower right")
axes[1, 1].grid(True, alpha=0.3)

# 6. Prediction Distribution
axes[1, 2].hist(y_train_pred_proba[y_train == 0], alpha=0.5, 
                label='Actual Low Risk (Train)', bins=30, color='blue')
axes[1, 2].hist(y_train_pred_proba[y_train == 1], alpha=0.5, 
                label='Actual High Risk (Train)', bins=30, color='red')
axes[1, 2].set_xlabel('Predicted Probability')
axes[1, 2].set_ylabel('Frequency')
axes[1, 2].set_title('Prediction Distribution (Train)', fontweight='bold')
axes[1, 2].legend()
axes[1, 2].axvline(x=0.5, color='black', linestyle='--', alpha=0.5)

plt.suptitle('NEURAL NETWORK OVERFITTING DEMONSTRATION\n'
             'Training Accuracy: ~95% | Test Accuracy: ~60%', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/neural_network_overfitting.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================
# DIAGNOSE THE PROBLEM
# ============================================

print("\n" + "="*60)
print("DIAGNOSING THE OVERFITTING PROBLEM")
print("="*60)

print("\n1. MODEL COMPLEXITY ISSUES:")
print("   • Too many layers (5 dense layers)")
print("   • Too many neurons (256 → 128 → 64 → 32 → 16)")
print("   • Training data: 6,400 samples, Parameters: ~80,000")
print("   • Parameter-to-sample ratio is too high")

print("\n2. TRAINING PATTERNS:")
print(f"   • Training accuracy plateaued at: {max(history.history['accuracy']):.3f}")
print(f"   • Validation accuracy peaked at: {max(history.history['val_accuracy']):.3f} (epoch {np.argmax(history.history['val_accuracy']) + 1})")
print(f"   • Then validation accuracy dropped to: {history.history['val_accuracy'][-1]:.3f}")

print("\n3. GENERALIZATION GAP:")
print(f"   • Training Accuracy: {train_accuracy:.3f}")
print(f"   • Test Accuracy:    {test_accuracy:.3f}")
print(f"   • Gap:             {train_accuracy - test_accuracy:.3f} → CLEAR OVERFITTING")

print("\n4. KEY OBSERVATION:")
print("   The model has memorized the training data noise instead of")
print("   learning generalizable patterns about diabetic retinopathy risk.")

# ============================================
# SAVE THE FAILED MODEL (FOR LEARNING PURPOSES)
# ============================================

overfit_model.save('../models/overfit_neural_network.h5')
print("\nOverfit model saved as 'models/overfit_neural_network.h5'")

# Save the predictions for comparison later
np.save('../data/overfit_train_predictions.npy', y_train_pred_proba)
np.save('../data/overfit_test_predictions.npy', y_test_pred_proba)

print("\n" + "="*60)
print("LESSON LEARNED:")
print("Complex models without proper regularization and validation")
print("can perform deceptively well on training data but fail to")
print("generalize to new, unseen clinical data.")
print("="*60)

MemoryError: 